## начальные настроуки


In [ ]:
SEED = 322
RUN_XLM_ROBERTA = True
RUN_RUBERT = True
RUN_TFIDF_WORD_SVC = True
RUN_TFIDF_CHAR_SVC = True
RUN_TEXTCNN = True

XLM_EPOCHS = 4
RUBERT_EPOCHS = 4
CNN_EPOCHS = 10
CNN_PATIENCE = 3

In [ ]:
# %pip -q install -U transformers datasets accelerate scikit-learn

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.3 requires scikit-learn<1.7,>=1.2, but you have scikit-learn 1.8.0 which is incompatible.
streamlit 1.45.1 requires protobuf<7,>=3.20, but you have protobuf 7.34.1 which is incompatible.


In [ ]:
import os, re, gc, ast, html, json, time, math, shutil, zipfile, random, warnings, unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
import torch, transformers, sklearn

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import hamming_loss, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

seed_everything(SEED)
print("CUDA:", torch.cuda.is_available(), "GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))


CUDA: False GPU count: 0


In [ ]:
ROOT = Path.cwd()

zip_path = ROOT / "2026-nlp.zip"
extract_dir = ROOT / "data"
extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

train = pd.read_csv(extract_dir / "train.csv", sep="\t")
test = pd.read_csv(extract_dir / "test.csv", sep="\t")
sample_sub = pd.read_csv(extract_dir / "sample_submission.csv")

print(train.shape, test.shape, sample_sub.shape)
display(train.head(3))
display(test.head(3))

(16701, 6) (4969, 5) (4969, 2)


,id,source,title,text,publication_date,target
0,0,Novosti,Рейтинг регионов по уровню закредитованности н...,Средний <content>уровень</content> <source>ria...,2019-12-23 00:00,"[0, 1, 0, 0, 0]"
1,1,Novosti,Названы самые закредитованные российские регионы,"МОСКВА, 23 дек — РИА Новости. Наиболее закре...",2019-12-23 00:21,"[0, 1, 0, 0, 0]"
2,2,Novosti,В России пройдут учения по обеспечению устойчи...,"МОСКВА, 23 дек - РИА Новости. Всероссийские у...",2019-12-23 00:28,"[1, 0, 0, 1, 0]"


,id,source,title,text,publication_date
0,16701,Spletnesti,Дым от австралийс...,...,2020-01-01 07:35
1,16702,Spletnesti,Во Владивостоке в...,...,2020-01-01 08:22
2,16703,Spletnesti,Папа римский шлёп...,...,2020-01-01 15:37


## EDA и делаем разметку

In [7]:
def parse_target(x):
    if isinstance(x, list):
        return [int(v) for v in x]
    if isinstance(x, np.ndarray):
        return [int(v) for v in x.tolist()]
    s = str(x).strip()
    try:
        v = ast.literal_eval(s)
        return [int(z) for z in v]
    except Exception:
        vals = [int(z) for z in re.findall(r"[01]", s)[:5]]
        return vals + [0] * (5 - len(vals))

y_all = np.array(train["target"].map(parse_target).tolist(), dtype=np.float32)
NUM_LABELS = y_all.shape[1]
LABEL_COLS = [f"label_{i}" for i in range(NUM_LABELS)]
for i, c in enumerate(LABEL_COLS):
    train[c] = y_all[:, i].astype(int)

print("Количество классов:", NUM_LABELS)
print("Среднее число меток на объект:", y_all.sum(axis=1).mean())
print("Доля объектов без меток:", (y_all.sum(axis=1) == 0).mean())
display(pd.DataFrame({"class": LABEL_COLS, "positive_count": y_all.sum(axis=0).astype(int), "positive_share": y_all.mean(axis=0)}))
display(train[LABEL_COLS].astype(int).apply(lambda r: str(r.tolist()), axis=1).value_counts().head(20).reset_index().rename(columns={"index":"target_combo", 0:"count"}))


Количество классов: 5
Среднее число меток на объект: 0.7810311
Доля объектов без меток: 0.3378839590443686


,class,positive_count,positive_share
0,label_0,7145,0.427819
1,label_1,2276,0.136279
2,label_2,1836,0.109934
3,label_3,1243,0.074427
4,label_4,544,0.032573


,target_combo,count
0,"[1, 0, 0, 0, 0]",5766
1,"[0, 0, 0, 0, 0]",5643
2,"[0, 0, 1, 0, 0]",1455
3,"[0, 1, 0, 0, 0]",1169
4,"[1, 1, 0, 0, 0]",689
5,"[0, 0, 0, 1, 0]",516
6,"[0, 0, 0, 0, 1]",303
7,"[1, 0, 0, 1, 0]",301
8,"[0, 1, 0, 1, 0]",246
9,"[1, 0, 1, 0, 0]",171


In [11]:
# очистка текста от мусорных симболов и эмодзи
_space_re = re.compile(r"\s+")
_url_re = re.compile(r"https?://\S+|www\.\S+")
_domain_re = re.compile(r"\b[a-zA-Zа-яА-Я0-9_-]+\.(?:ru|com|org|net|рф)\b", flags=re.I)

_emoji_re = re.compile(
    "["
    "\U0001F300-\U0001F5FF"
    "\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F"
    "\U0001F780-\U0001F7FF"
    "\U0001F800-\U0001F8FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA00-\U0001FAFF"
    "\u2300-\u23FF"
    "\u2600-\u26FF"
    "\u2700-\u27BF"
    "\u2B00-\u2BFF"
    "\uFE0E-\uFE0F"
    "]+",
    flags=re.UNICODE,
)

extra_symbol_codes = [
    "\u203c",  # double exclamation mark
    "\u2049",  # exclamation question mark
    "\u2620",  # skull and crossbones
    "\u2622",  # radioactive sign
    "\u2623",  # biohazard sign
    "\u00a9",  # copyright
    "\u00ae",  # registered sign
    "\u2122",  # trademark sign
    "\u00a7",  # section sign
]

def clean_text(x):
    if pd.isna(x):
        return ""

    s = str(x)
    for _ in range(6):
        s = html.unescape(s)

    s = (
        s.replace("\xa0", " ")
         .replace("\u200b", " ")
         .replace("\ufeff", " ")
         .replace("\u200c", " ")
         .replace("\u200d", " ")
    )

    s = s.replace("<![CDATA[", " ")
    s = s.replace("]]>", " ")
    s = re.sub(r"<!--.*?-->", " ", s, flags=re.DOTALL)
    s = re.sub(r"<\s*source[^>]*>.*?<\s*/\s*source\s*>", " ", s, flags=re.I | re.S)

    for _ in range(5):
        s = re.sub(r"<[^>]*>", " ", s)

    s = re.sub(r'\bclass\s*=\s*["\'][^"\']*["\']', " ", s, flags=re.I)
    s = re.sub(r'\bstyle\s*=\s*["\'][^"\']*["\']', " ", s, flags=re.I)
    s = re.sub(r'\bid\s*=\s*["\'][^"\']*["\']', " ", s, flags=re.I)
    s = re.sub(r'\bhref\s*=\s*["\'][^"\']*["\']', " ", s, flags=re.I)

    s = re.sub(
        r"\b/?(?:span|div|p|br|strong|em|u|i|b|a|section|article|doc|content|source|url|hr|li|ul|ol|blockquote)\b",
        " ",
        s,
        flags=re.I,
    )

    s = s.replace("<", " ").replace(">", " ")
    s = _url_re.sub(" ", s)
    s = _domain_re.sub(" ", s)

    s = re.sub(
        r"&\s*(?:nbsp|amp|lt|gt|quot|apos|copy|reg|laquo|raquo|mdash|ndash)\s*;?",
        " ",
        s,
        flags=re.I,
    )
    s = re.sub(r"&\s*#\s*x?[0-9a-fA-F]+\s*;?", " ", s, flags=re.I)

    s = re.sub(
        r"\b(?:nbsp|amp|lt|gt|quot|apos|copy|reg|laquo|raquo|mdash|ndash);?",
        " ",
        s,
        flags=re.I,
    )

    s = s.replace("&", " ")
    s = _emoji_re.sub(" ", s)

    for bad in extra_symbol_codes:
        s = s.replace(bad, " ")

    cleaned_chars = []
    for ch in s:
        cat = unicodedata.category(ch)
        if cat.startswith(("Cc", "Cs", "Cf")):
            continue
        if cat == "So":
            continue
        cleaned_chars.append(ch)

    s = "".join(cleaned_chars)
    s = re.sub(r"\s+'\s*", " ", s)
    s = re.sub(r'\s+"\s*', " ", s)
    s = re.sub(r"\s+([,.;:!?%)])", r"\1", s)
    s = re.sub(r"([(])\s+", r"\1", s)
    s = _space_re.sub(" ", s).strip()

    return s


def head_tail_text(s, max_chars=9000, head_ratio=0.72):
    s = str(s)
    if len(s) <= max_chars:
        return s
    head = int(max_chars * head_ratio)
    tail = max_chars - head
    return s[:head] + " ... " + s[-tail:]


USE_SOURCE_IN_TEXT = False
USE_DATE_IN_TEXT = False


def build_model_text(df, max_chars=9000):
    title = df["title"].fillna("").map(clean_text)
    text = df["text"].fillna("").map(clean_text)

    parts = []
    if USE_SOURCE_IN_TEXT and "source" in df.columns:
        parts.append("Источник: " + df["source"].fillna("").map(clean_text))
    if USE_DATE_IN_TEXT and "publication_date" in df.columns:
        parts.append("Дата: " + df["publication_date"].fillna("").map(clean_text))

    parts.append("Заголовок: " + title)
    parts.append("Текст: " + text)

    out = parts[0]
    for p in parts[1:]:
        out = out + "\n" + p

    return out.map(lambda z: head_tail_text(z, max_chars=max_chars))

train_texts = build_model_text(train)
test_texts = build_model_text(test)

print("train_texts shape:", train_texts.shape)
print("test_texts shape:", test_texts.shape)

train_texts shape: (16701,)
test_texts shape: (4969,)


In [12]:
#смотрим 3 случайных очищенных текста из train
rng = np.random.default_rng(SEED)
random_indices = rng.choice(len(train_texts), size=3, replace=False)

for n, idx in enumerate(random_indices, start=1):
    print("=" * 120)
    print(f"RANDOM CLEANED TRAIN TEXT #{n}")
    print("INDEX:", idx)
    
    if "id" in train.columns:
        print("ID:", train.loc[idx, "id"])
    
    if "source" in train.columns:
        print("SOURCE:", train.loc[idx, "source"])
    
    if "title" in train.columns:
        print("RAW TITLE:", train.loc[idx, "title"])
    
    if "target" in train.columns:
        print("TARGET:", train.loc[idx, "target"])
    
    print("-" * 120)
    print(train_texts.iloc[idx][:2000])
    print()

RANDOM CLEANED TRAIN TEXT #1
INDEX: 4933
ID: 4933
SOURCE: Novosti
RAW TITLE: В штате Вашингтон объявили режим ЧП из-за коронавируса
TARGET: [0, 0, 0, 0, 0]
------------------------------------------------------------------------------------------------------------------------
Заголовок: В штате Вашингтон объявили режим ЧП из-за коронавируса
Текст: МОСКВА, 1 мар - РИА Новости. Губернатор штата Вашингтон Джей Инсли объявил …о введении чрезвычайного положения ”на территории штата в связи с распространением нового коронавируса и первой зафиксированной в стране смертью от COVID-19, сообщается в заявлении, опубликованном на сайте губернатора. Я, Джей Инсли, …губернатор штата Вашингтон в свете сложившейся ситуации … заявляю, что во всех округах штата Вашингтон »действует чрезвычайное положение", ”- говорится в распространенном в субботу постановлении. – … Губернатор также предписал государственным учреждениям и департаментам использовать госресурсы и делать все возможное для борьбы с распрост

In [13]:
#смотрим 3 случайных очищенных текста из test
random_test_indices = rng.choice(len(test_texts), size=3, replace=False)

for n, idx in enumerate(random_test_indices, start=1):
    print("=" * 120)
    print(f"RANDOM CLEANED TEST TEXT #{n}")
    print("INDEX:", idx)
    
    if "id" in test.columns:
        print("ID:", test.loc[idx, "id"])
    
    if "source" in test.columns:
        print("SOURCE:", test.loc[idx, "source"])
    
    if "title" in test.columns:
        print("RAW TITLE:", test.loc[idx, "title"])
    
    print("-" * 120)
    print(test_texts.iloc[idx][:2000])
    print()

RANDOM CLEANED TEST TEXT #1
INDEX: 1368
ID: 18069
SOURCE: Zholtosti
RAW TITLE: Путин предложил выплатить всем потерявшим работу по три максимальных пособия по безработице
------------------------------------------------------------------------------------------------------------------------
Заголовок: Путин предложил выплатить всем потерявшим работу по три максимальных пособия по безработице
Текст: Президент России Владимир Путин объявил о мерах поддержки граждан, потерявших работу на фоне пандемии коронавирусной инфекции. Президент предложил, чтобы каждый россиян, потерявший работу и обратившийся в службу занятости после 1 марта, в апреле-июне получил три максимальных пособия по безработице в максимальном размере — 12130 рублей. Путин попросил, чтобы россияне могли оформить пособие максимально просто и дистанционно. Также президент предложил, чтобы семьи, где родители остались без работы, получали в апреле-июне по 3000 рублей на каждого несовершеннолетнего ребенка. Кроме того, Путин п

## разделение данных и пороги

In [ ]:
target_combo = train[LABEL_COLS].astype(str).agg("".join, axis=1)
counts = target_combo.value_counts()
stratify = target_combo.where(target_combo.map(counts) >= 2, "RARE")

train_idx, val_idx = train_test_split(np.arange(len(train)), test_size=0.20, random_state=SEED, shuffle=True, stratify=stratify)
y_train = y_all[train_idx]
y_val = y_all[val_idx]

print("train split:", len(train_idx), "val split:", len(val_idx), "test:", len(test))
print("train shares:", np.round(y_train.mean(axis=0), 4))
print("val shares:", np.round(y_val.mean(axis=0), 4))

pos = y_train.sum(axis=0)
neg = len(y_train) - pos
pos_weight = np.clip(neg / np.maximum(pos, 1), 1.0, 20.0)
print("pos_weight:", pos_weight)


def sigmoid_np(x):
    return 1 / (1 + np.exp(-x))


def tune_thresholds(y_true, probs, grid=None):
    if grid is None:
        grid = np.arange(0.05, 0.96, 0.01)
    thresholds = []
    preds = np.zeros_like(probs, dtype=int)
    for j in range(y_true.shape[1]):
        best_t, best_score = 0.5, 999
        for t in grid:
            p = (probs[:, j] >= t).astype(int)
            score = hamming_loss(y_true[:, j], p)
            if score < best_score:
                best_score, best_t = score, float(t)
        thresholds.append(best_t)
        preds[:, j] = (probs[:, j] >= best_t).astype(int)
    return np.array(thresholds), preds


def format_target(row):
    return "[" + ",".join(str(int(v)) for v in row) + "]"


train split: 13360 val split: 3341 test: 4969
train shares: [0.4278 0.1362 0.1099 0.0745 0.0326]
val shares: [0.4277 0.1368 0.1101 0.0742 0.0323]
pos_weight: [ 1.3372988  6.344695   8.100818  12.427135  20.       ]


## токенизациия, батчи, метрики, вероятности

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_length=512):
        self.texts = list(texts)
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), truncation=True, max_length=self.max_length, padding=False)
        if self.labels is not None:
            enc["labels"] = self.labels[idx].astype(np.float32)
        return enc

class MultilabelDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
    def __call__(self, features):
        labels = None
        if "labels" in features[0]:
            labels = [f.pop("labels") for f in features]
        batch = self.tokenizer.pad(features, padding=True, return_tensors="pt")
        if labels is not None:
            batch["labels"] = torch.tensor(np.array(labels), dtype=torch.float32)
        return batch

class WeightedMultilabelTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = torch.tensor(pos_weight, dtype=torch.float32) if pos_weight is not None else None
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        pw = self.pos_weight.to(logits.device) if self.pos_weight is not None else None
        loss = torch.nn.BCEWithLogitsLoss(pos_weight=pw)(logits, labels.float())
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = sigmoid_np(logits)
    th, pred = tune_thresholds(labels, probs)
    return {"hamming_loss": hamming_loss(labels, pred), "f1_micro": f1_score(labels, pred, average="micro", zero_division=0), "f1_macro": f1_score(labels, pred, average="macro", zero_division=0)}

PROB_DIR = ROOT / "saved_probabilities_nlp_plus"
PROB_DIR.mkdir(exist_ok=True)


def make_args(output_dir, cfg, n_train):
    steps_per_epoch = math.ceil(n_train / max(1, cfg["per_device_train_batch_size"] * max(1, torch.cuda.device_count()) * cfg["gradient_accumulation_steps"]))
    warmup_steps = max(20, int(0.06 * steps_per_epoch * cfg["epochs"]))
    params = dict(output_dir=str(output_dir), seed=SEED, data_seed=SEED, num_train_epochs=cfg["epochs"], learning_rate=cfg["lr"], per_device_train_batch_size=cfg["per_device_train_batch_size"], per_device_eval_batch_size=cfg["per_device_eval_batch_size"], gradient_accumulation_steps=cfg["gradient_accumulation_steps"], warmup_steps=warmup_steps, weight_decay=0.01, lr_scheduler_type="cosine", logging_steps=50, save_strategy="epoch", eval_strategy="epoch", save_total_limit=1, load_best_model_at_end=True, metric_for_best_model="hamming_loss", greater_is_better=False, fp16=torch.cuda.is_available(), dataloader_num_workers=0, report_to="none", remove_unused_columns=False)
    try:
        return TrainingArguments(**params)
    except TypeError:
        params["evaluation_strategy"] = params.pop("eval_strategy")
        return TrainingArguments(**params)


def predict_transformer_probs(trainer, ds):
    return sigmoid_np(trainer.predict(ds).predictions)


def train_one_transformer(cfg):
    tag = cfg["tag"]
    val_path = PROB_DIR / f"{tag}_val_probs.npy"
    test_path = PROB_DIR / f"{tag}_test_probs.npy"
    if val_path.exists() and test_path.exists():
        print("Loading cached:", tag)
        return np.load(val_path), np.load(test_path)

    print("\n" + "="*80)
    print("Training", tag, cfg["model_name"])
    print("="*80)
    clear_memory(); seed_everything(SEED)

    tokenizer = AutoTokenizer.from_pretrained(cfg["model_name"], use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(cfg["model_name"], num_labels=NUM_LABELS, problem_type="multi_label_classification", use_safetensors=True)
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
        print("Gradient checkpointing enabled")
    if hasattr(model.config, "use_cache"):
        model.config.use_cache = False

    train_ds = TextDataset(train_texts.iloc[train_idx].values, y_train, tokenizer, cfg["max_length"])
    val_ds = TextDataset(train_texts.iloc[val_idx].values, y_val, tokenizer, cfg["max_length"])
    test_ds = TextDataset(test_texts.values, None, tokenizer, cfg["max_length"])
    collator = MultilabelDataCollator(tokenizer)
    print("Debug batch:", {k: tuple(v.shape) for k, v in collator([train_ds[0], train_ds[1]]).items()})

    args = make_args(ROOT / f"runs/{tag}", cfg, len(train_ds))
    kwargs = dict(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, data_collator=collator, compute_metrics=compute_metrics, pos_weight=pos_weight)
    import inspect
    sig = inspect.signature(Trainer.__init__)
    if "processing_class" in sig.parameters:
        kwargs["processing_class"] = tokenizer
    elif "tokenizer" in sig.parameters:
        kwargs["tokenizer"] = tokenizer
    trainer = WeightedMultilabelTrainer(**kwargs)
    print(trainer.train())

    val_probs = predict_transformer_probs(trainer, val_ds)
    test_probs = predict_transformer_probs(trainer, test_ds)
    th, pred = tune_thresholds(y_val, val_probs)
    print(tag, "val hamming:", hamming_loss(y_val, pred), "thresholds:", np.round(th, 3))
    np.save(val_path, val_probs); np.save(test_path, test_probs)
    shutil.rmtree(ROOT / f"runs/{tag}", ignore_errors=True)
    del trainer, model, tokenizer, train_ds, val_ds, test_ds, collator
    clear_memory()
    return val_probs, test_probs


## обучение моделек в ансамбль

In [ ]:
MODEL_CONFIGS = [
    {
        "tag": "xlm_roberta_large",
        "model_name": "xlm-roberta-large",
        "enabled": RUN_XLM_ROBERTA,
        "max_length": 512,
        "epochs": XLM_EPOCHS,
        "lr": 1e-5,
        "per_device_train_batch_size": 2,
        "per_device_eval_batch_size": 4,
        "gradient_accumulation_steps": 8,
    },
    {
        "tag": "rubert_deeppavlov",
        "model_name": "DeepPavlov/rubert-base-cased",
        "enabled": RUN_RUBERT,
        "max_length": 512,
        "epochs": RUBERT_EPOCHS,
        "lr": 2e-5,
        "per_device_train_batch_size": 4,
        "per_device_eval_batch_size": 8,
        "gradient_accumulation_steps": 4,
    },
]

val_prob_sources, test_prob_sources = {}, {}
for cfg in MODEL_CONFIGS:
    if not cfg["enabled"]:
        continue
    try:
        vp, tp = train_one_transformer(cfg)
        val_prob_sources[cfg["tag"]] = vp
        test_prob_sources[cfg["tag"]] = tp
    except Exception as e:
        print("Model failed and will be skipped:", cfg["tag"], repr(e))
        clear_memory()

print("Neural sources:", list(val_prob_sources.keys()))



Training xlm_roberta_large xlm-roberta-large


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=3340, training_loss=1.772014372363062, metrics={'train_runtime': 2168.8027, 'train_samples_per_second': 24.64, 'train_steps_per_second': 1.54, 'total_flos': 1.1368712085891576e+16, 'train_loss': 1.772014372363062, 'epoch': 4.0})


rubert_deeppavlov val hamming: 0.046034121520502846 thresholds: [0.46 0.88 0.76 0.95 0.88]
Neural sources: ['xlm_roberta_large', 'rubert_deeppavlov']


In [ ]:
# TF-IDF word + LinearSVC
def train_tfidf_svc_word():
    tag = "tfidf_word_linearsvc"
    val_path, test_path = PROB_DIR / f"{tag}_val_probs.npy", PROB_DIR / f"{tag}_test_probs.npy"
    if val_path.exists() and test_path.exists():
        print("Loading cached:", tag)
        return np.load(val_path), np.load(test_path)
    vec = TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=2, max_df=0.95, max_features=250000, sublinear_tf=True, lowercase=True)
    Xtr = vec.fit_transform(train_texts.iloc[train_idx])
    Xv = vec.transform(train_texts.iloc[val_idx])
    Xt = vec.transform(test_texts)
    best = (999, None, None, None)
    for C in [0.5, 1.0, 2.0, 4.0]:
        print("C=", C)
        dv, dt = [], []
        for j in range(NUM_LABELS):
            clf = LinearSVC(C=C, class_weight="balanced", random_state=SEED, max_iter=5000)
            clf.fit(Xtr, y_train[:,j].astype(int))
            dv.append(clf.decision_function(Xv)); dt.append(clf.decision_function(Xt))
        dv, dt = np.vstack(dv).T, np.vstack(dt).T
        for scale in [0.75, 1.0, 1.5, 2.0, 2.5]:
            vp, tp = sigmoid_np(dv/scale), sigmoid_np(dt/scale)
            th, pred = tune_thresholds(y_val, vp)
            score = hamming_loss(y_val, pred)
            if score < best[0]:
                best = (score, vp, tp, (C, scale, th))
    print("Best TF-IDF word val:", best[0], "info:", best[3][0], best[3][1], np.round(best[3][2],3))
    np.save(val_path, best[1]); np.save(test_path, best[2])
    del vec, Xtr, Xv, Xt
    clear_memory()
    return best[1], best[2]

if RUN_TFIDF_WORD_SVC:
    vp, tp = train_tfidf_svc_word()
    val_prob_sources["tfidf_word_linearsvc"] = vp
    test_prob_sources["tfidf_word_linearsvc"] = tp
print("Sources:", list(val_prob_sources.keys()))


Loading cached: tfidf_word_linearsvc
Sources: ['xlm_roberta_large', 'rubert_deeppavlov', 'tfidf_word_linearsvc', 'tfidf_charwb_linearsvc', 'textcnn_conv1d']


In [ ]:
# Char_wb TF-IDF + LinearSVC
def train_tfidf_svc_char():
    tag = "tfidf_charwb_linearsvc"
    val_path, test_path = PROB_DIR / f"{tag}_val_probs.npy", PROB_DIR / f"{tag}_test_probs.npy"
    if val_path.exists() and test_path.exists():
        print("Loading cached:", tag)
        return np.load(val_path), np.load(test_path)

    vec = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_df=0.98,
        max_features=300000,
        sublinear_tf=True,
        lowercase=True,
        dtype=np.float32,
    )

    Xtr = vec.fit_transform(train_texts.iloc[train_idx])
    Xv = vec.transform(train_texts.iloc[val_idx])
    Xt = vec.transform(test_texts)

    best = (999, None, None, None)
    for C in [0.5, 1.0, 2.0, 4.0]:
        print("char_wb C=", C)
        dv, dt = [], []
        for j in range(NUM_LABELS):
            clf = LinearSVC(C=C, class_weight="balanced", random_state=SEED, max_iter=7000)
            clf.fit(Xtr, y_train[:, j].astype(int))
            dv.append(clf.decision_function(Xv))
            dt.append(clf.decision_function(Xt))

        dv, dt = np.vstack(dv).T, np.vstack(dt).T

        for scale in [0.75, 1.0, 1.5, 2.0, 2.5]:
            vp, tp = sigmoid_np(dv / scale), sigmoid_np(dt / scale)
            th, pred = tune_thresholds(y_val, vp)
            sc = hamming_loss(y_val, pred)
            print(f"  scale={scale}: val_hamming={sc:.6f}, th={np.round(th,3)}")
            if sc < best[0]:
                best = (sc, vp, tp, (C, scale, th))

    print("Best char_wb SVC:", best[0], "params:", best[3])
    np.save(val_path, best[1])
    np.save(test_path, best[2])

    del vec, Xtr, Xv, Xt
    clear_memory()
    return best[1], best[2]

if RUN_TFIDF_CHAR_SVC:
    vp, tp = train_tfidf_svc_char()
    val_prob_sources["tfidf_charwb_linearsvc"] = vp
    test_prob_sources["tfidf_charwb_linearsvc"] = tp


Loading cached: tfidf_charwb_linearsvc


In [ ]:
# TextCNN и Conv1D
def simple_tokenize(text):
    return re.findall(r"[а-яА-ЯёЁa-zA-Z0-9]+", str(text).lower())

class TextCNNDataset(Dataset):
    def __init__(self, texts, labels=None, vocab=None, max_len=1050):
        self.texts, self.labels, self.vocab, self.max_len = list(texts), labels, vocab, max_len
    def __len__(self):
        return len(self.texts)
    def encode(self, text):
        ids = [self.vocab.get(t, 1) for t in simple_tokenize(text)][:self.max_len]
        ids += [0] * (self.max_len - len(ids))
        return ids
    def __getitem__(self, idx):
        item = {"input_ids": torch.tensor(self.encode(self.texts[idx]), dtype=torch.long)}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return item

class TextCNN(nn.Module):
    def __init__(self, vocab_size, num_labels, emb_dim=300, filters=256, kernels=(2,3,4,5,7), dropout=0.42):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(emb_dim, filters, k) for k in kernels])
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(filters * len(kernels), num_labels)
    def forward(self, input_ids):
        x = self.embedding(input_ids).transpose(1, 2)
        x = torch.cat([torch.relu(conv(x)).max(dim=2).values for conv in self.convs], dim=1)
        return self.classifier(self.dropout(x))


def predict_cnn(model, loader, device):
    model.eval()
    probs = []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch["input_ids"].to(device))
            probs.append(torch.sigmoid(logits).detach().cpu().numpy())
    return np.vstack(probs)


def train_textcnn():
    tag = "textcnn_conv1d"
    val_path, test_path = PROB_DIR / f"{tag}_val_probs.npy", PROB_DIR / f"{tag}_test_probs.npy"
    if val_path.exists() and test_path.exists():
        print("Loading cached:", tag)
        return np.load(val_path), np.load(test_path)

    print("Training stronger TextCNN / Conv1D...")
    seed_everything(SEED)
    clear_memory()

    counter = Counter()
    for text in train_texts.iloc[train_idx]:
        counter.update(simple_tokenize(text))

    vocab = {"<PAD>": 0, "<UNK>": 1}
    for w, _ in counter.most_common(110000 - 2):
        vocab[w] = len(vocab)
    print("Vocab size:", len(vocab))

    tr_ds = TextCNNDataset(train_texts.iloc[train_idx].values, y_train, vocab)
    va_ds = TextCNNDataset(train_texts.iloc[val_idx].values, y_val, vocab)
    te_ds = TextCNNDataset(test_texts.values, None, vocab)

    tr = DataLoader(tr_ds, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
    va = DataLoader(va_ds, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
    te = DataLoader(te_ds, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = TextCNN(len(vocab), NUM_LABELS).to(device)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)

    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, dtype=torch.float32).to(device))
    opt = torch.optim.AdamW(model.parameters(), lr=1.6e-3, weight_decay=2e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CNN_EPOCHS)

    best_score, best_state = 999, None
    bad_epochs = 0

    for epoch in range(1, CNN_EPOCHS + 1):
        model.train()
        total = 0.0

        for batch in tr:
            opt.zero_grad(set_to_none=True)
            logits = model(batch["input_ids"].to(device))
            loss = criterion(logits, batch["labels"].to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += float(loss.item())

        scheduler.step()

        vp = predict_cnn(model, va, device)
        th, pred = tune_thresholds(y_val, vp)
        score = hamming_loss(y_val, pred)
        print(f"TextCNN epoch {epoch}: loss={total/max(1,len(tr)):.5f}, val_hamming={score:.6f}, th={np.round(th,3)}")

        if score + 1e-8 < best_score:
            best_score = score
            bad_epochs = 0
            m = model.module if isinstance(model, nn.DataParallel) else model
            best_state = {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= CNN_PATIENCE:
                print(f"Early stopping TextCNN at epoch {epoch}; best={best_score:.6f}")
                break

    m = model.module if isinstance(model, nn.DataParallel) else model
    m.load_state_dict(best_state)
    vp, tp = predict_cnn(model, va, device), predict_cnn(model, te, device)
    print("Best TextCNN val:", best_score)

    np.save(val_path, vp)
    np.save(test_path, tp)

    del model, tr_ds, va_ds, te_ds, tr, va, te
    clear_memory()
    return vp, tp

if RUN_TEXTCNN:
    vp, tp = train_textcnn()
    val_prob_sources["textcnn_conv1d"] = vp
    test_prob_sources["textcnn_conv1d"] = tp

print("All sources:", list(val_prob_sources.keys()))


Loading cached: textcnn_conv1d
All sources: ['xlm_roberta_large', 'rubert_deeppavlov', 'tfidf_word_linearsvc', 'tfidf_charwb_linearsvc', 'textcnn_conv1d']


## поиск лучшего ансамбля

In [ ]:
source_names = list(val_prob_sources.keys())
assert len(source_names) > 0, "No sources available"
print("Sources:", source_names)


def blend_probs(weights, names):
    val = np.zeros((len(val_idx), NUM_LABELS), dtype=float)
    tst = np.zeros((len(test), NUM_LABELS), dtype=float)
    for w, n in zip(weights, names):
        val += w * val_prob_sources[n]
        tst += w * test_prob_sources[n]
    return val, tst


def evaluate_weights(weights, names):
    vp, _ = blend_probs(weights, names)
    th, pred = tune_thresholds(y_val, vp)
    return hamming_loss(y_val, pred), th, pred

rows=[]
# по одной
for i,n in enumerate(source_names):
    w = np.zeros(len(source_names)); w[i]=1
    sc, th, pred = evaluate_weights(w, source_names)
    rows.append({"kind":f"single_{n}", "score":sc, "weights":w, "thresholds":th, "all_zero":float((pred.sum(1)==0).mean())})
# попарно
for i in range(len(source_names)):
    for j in range(i+1, len(source_names)):
        for a in np.arange(0.05,0.96,0.05):
            w = np.zeros(len(source_names)); w[i]=a; w[j]=1-a
            sc, th, pred = evaluate_weights(w, source_names)
            rows.append({"kind":f"pair_{source_names[i]}_{source_names[j]}", "score":sc, "weights":w, "thresholds":th, "all_zero":float((pred.sum(1)==0).mean())})
# микс
rng = np.random.default_rng(SEED)
if len(source_names) >= 3:
    for _ in range(5000):
        w = rng.dirichlet(np.ones(len(source_names)))
        ok=True
        # не даем быть слишком большими чтобы не переобучаться
        for aux,max_w in [("textcnn_conv1d",0.25),("tfidf_word_linearsvc",0.40),("tfidf_charwb_linearsvc",0.35)]:
            if aux in source_names and w[source_names.index(aux)] > max_w:
                ok=False; break
        if not ok: continue
        sc, th, pred = evaluate_weights(w, source_names)
        rows.append({"kind":"random", "score":sc, "weights":w, "thresholds":th, "all_zero":float((pred.sum(1)==0).mean())})

ens_df = pd.DataFrame(rows).sort_values("score").reset_index(drop=True)
display(ens_df.head(20)[["kind","score","thresholds","all_zero"]])

best = ens_df.iloc[0]
best_weights, best_thresholds = best["weights"], best["thresholds"]
print("Best ensemble:", best["score"], best["kind"])
for n,w in zip(source_names,best_weights): print(f"  {n}: {w:.4f}")
print("thresholds:", np.round(best_thresholds,4))

best_val_probs, best_test_probs = blend_probs(best_weights, source_names)
best_val_pred = (best_val_probs >= best_thresholds).astype(int)
best_test_pred = (best_test_probs >= best_thresholds).astype(int)
print("val true shares:", np.round(y_val.mean(0),4))
print("val pred shares:", np.round(best_val_pred.mean(0),4))
print("test pred shares:", np.round(best_test_pred.mean(0),4))
print("test all-zero:", float((best_test_pred.sum(1)==0).mean()))


Sources: ['xlm_roberta_large', 'rubert_deeppavlov', 'tfidf_word_linearsvc', 'tfidf_charwb_linearsvc', 'textcnn_conv1d']


,kind,score,thresholds,all_zero
0,random,0.040766,"[0.4700000000000001, 0.8200000000000002, 0.710...",0.346004
1,random,0.040766,"[0.4700000000000001, 0.8200000000000002, 0.710...",0.346304
2,random,0.040826,"[0.4600000000000001, 0.8100000000000002, 0.730...",0.346902
3,random,0.040826,"[0.4700000000000001, 0.8400000000000002, 0.720...",0.347800
4,random,0.040826,"[0.4700000000000001, 0.8600000000000002, 0.740...",0.348698
5,random,0.040886,"[0.4700000000000001, 0.7800000000000001, 0.700...",0.343909
6,random,0.040946,"[0.45000000000000007, 0.8400000000000002, 0.73...",0.344508
7,random,0.040946,"[0.4600000000000001, 0.7900000000000001, 0.710...",0.346004
8,random,0.041006,"[0.4800000000000001, 0.7700000000000002, 0.670...",0.344807
9,random,0.041006,"[0.4800000000000001, 0.7700000000000002, 0.660...",0.345705


Best ensemble: 0.04076623765339719 random
  xlm_roberta_large: 0.5599
  rubert_deeppavlov: 0.1730
  tfidf_word_linearsvc: 0.0295
  tfidf_charwb_linearsvc: 0.1851
  textcnn_conv1d: 0.0525
thresholds: [0.47 0.82 0.71 0.81 0.73]
val true shares: [0.4277 0.1368 0.1101 0.0742 0.0323]
val pred shares: [0.4361 0.1239 0.0934 0.0608 0.0224]
test pred shares: [0.4613 0.0896 0.1258 0.0644 0.0203]
test all-zero: 0.30388408130408534


## формируем сабмисион

In [ ]:
FINAL_WEIGHTS_DICT = {
    "xlm_roberta_large": 0.5599,
    "rubert_deeppavlov": 0.1730,
    "tfidf_word_linearsvc": 0.0295,
    "tfidf_charwb_linearsvc": 0.1851,
    "textcnn_conv1d": 0.0525,
}

FINAL_THRESHOLDS = np.array([0.47, 0.82, 0.71, 0.81, 0.73])
missing_sources = [name for name in FINAL_WEIGHTS_DICT if name not in source_names]

assert len(missing_sources) == 0, f"Missing probability sources: {missing_sources}"

final_weights = np.array(
    [FINAL_WEIGHTS_DICT.get(name, 0.0) for name in source_names],
    dtype=float
)
final_weights = final_weights / final_weights.sum()

print("Final ensemble weights:")
for name, weight in zip(source_names, final_weights):
    print(f"  {name}: {weight:.4f}")
print("Final thresholds:", np.round(FINAL_THRESHOLDS, 4))


final_val_probs, final_test_probs = blend_probs(final_weights, source_names)
final_val_pred = (final_val_probs >= FINAL_THRESHOLDS).astype(int)
final_test_pred = (final_test_probs >= FINAL_THRESHOLDS).astype(int)

print("Final validation hamming loss:", hamming_loss(y_val, final_val_pred))
print("val true shares:", np.round(y_val.mean(axis=0), 4))
print("val pred shares:", np.round(final_val_pred.mean(axis=0), 4))
print("test pred shares:", np.round(final_test_pred.mean(axis=0), 4))
print("test all-zero share:", float((final_test_pred.sum(axis=1) == 0).mean()))


# формируем submission в нужном формате
pred_df = pd.DataFrame({
    "id": test["id"].values,
    "target": [format_target(row) for row in final_test_pred]
})

submission = sample_sub[["id"]].merge(pred_df, on="id", how="left")
assert submission["target"].isna().sum() == 0
assert len(submission) == len(sample_sub)

OUTPUT_PATH = ROOT / "sample_submission.csv"
submission.to_csv(OUTPUT_PATH, index=False)

print("Saved final submission to:", OUTPUT_PATH)
display(submission.head())